In [ ]:
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import cell2cell as c2c
from collections import defaultdict

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# DIRECTORY SETUP
# ---------------------------------------------------------
INPUT_DIR = './results/tensor_outputs/'
OUTPUT_FIGS = './results/figures/tensor/'
os.makedirs(OUTPUT_FIGS, exist_ok=True)

# ---------------------------------------------------------
# HELPER FUNCTION
# ---------------------------------------------------------
def get_context_dict():
    """Returns a dictionary mapping sample IDs to their disease groups."""
    context_dict = defaultdict(lambda: 'Unknown')
    mapping = {
        'H08': 'HCC', 'H70': 'HCC', 'H72': 'HCC', 'H73a': 'HCC', 'H74': 'HCC',
        'H73b': 'HCC', 'H75': 'HCC', 'H49b': 'HCC', 'H77': 'HCC', 'H68a': 'HCC',
        'H58c': 'HCC', 'H68b': 'HCC', 'H55': 'HCC', 'H62': 'HCC', 'H34b': 'HCC',
        'H34a': 'HCC', 'H34c': 'HCC', 'H41': 'HCC', 'H43': 'HCC', 'H49a': 'HCC',
        'H58a': 'HCC', 'H58b': 'HCC', 'H63': 'HCC', 'H65': 'HCC', 'H21': 'HCC',
        'H28': 'HCC', 'H23': 'HCC', 'H30': 'HCC', 'H38': 'HCC', 'H18': 'HCC',
        'H37': 'HCC',
        'C76': 'ICCa', 'C60': 'ICCa', 'C56': 'ICCa', 'C26a': 'ICCa', 'C42': 'ICCa',
        'C46b': 'ICCa', 'C66': 'ICCa', 'C26b': 'ICCa', 'C25': 'ICCa', 'C29': 'ICCa',
        'C39': 'ICCa', 'C35': 'ICCa'
    }
    context_dict.update(mapping)
    return context_dict

# ---------------------------------------------------------
# GLOBAL VISUALIZATION FUNCTIONS
# ---------------------------------------------------------
def plot_global_tensor_visualizations(factors, context_dict, output_dir):
    """
    Generate global visualizations for the tensor decomposition (Boxplots, Context Clustermap, LR Clustermap).
    """
    groups_order = ['HCC', 'ICCa']

    # 1. Context Boxplots (Evaluating statistical significance between diseases)
    print("[*] Generating Context Loadings Boxplots (with independent t-test)...")
    _ = c2c.plotting.context_boxplot(
        context_loadings=factors['Contexts'],
        metadict=context_dict,
        nrows=3,
        figsize=(16, 12),
        group_order=groups_order,
        statistical_test='t-test_ind',
        pval_correction='fdr_bh',
        cmap='plasma',
        verbose=False,
        filename=os.path.join(output_dir, 'Boxplots_Context_Loadings.pdf'),
        text_format='full'
    )

    # 2. Context Clustermap
    print("[*] Generating Context Clustermap...")
    condition_colors = c2c.plotting.aesthetics.get_colors_from_labels(
        groups_order, cmap='plasma'
    )
    color_dict = {k: condition_colors[v] for k, v in context_dict.items()}
    col_colors = pd.Series(color_dict).to_frame(name='Disease type')

    sample_cm = c2c.plotting.loading_clustermap(
        factors['Contexts'],
        use_zscore=False,
        col_colors=col_colors,
        figsize=(16, 6),
        dendrogram_ratio=0.3,
        cbar_fontsize=12,
        tick_fontsize=14,
        filename=os.path.join(output_dir, 'Clustermap_Contexts.pdf')
    )

    # Add legend to Context Clustermap
    plt.sca(sample_cm.ax_heatmap)
    c2c.plotting.aesthetics.generate_legend(
        color_dict=condition_colors,
        bbox_to_anchor=(1.1, 0.5),
        title='Disease type'
    )
    plt.savefig(os.path.join(output_dir, 'Clustermap_Contexts_with_Legend.pdf'), bbox_inches='tight')
    plt.close()

    # 3. Ligand-Receptor Clustermap
    print("[*] Generating Ligand-Receptor Loadings Clustermap...")
    c2c.plotting.loading_clustermap(
        factors['Ligand-Receptor Pairs'],
        loading_threshold=0.1, # Display only top LRs
        use_zscore=False,
        figsize=(28, 8),
        filename=os.path.join(output_dir, 'Clustermap_LR_Pairs.pdf'),
        row_cluster=False # Keep factors ordered sequentially
    )
    plt.close()

# ---------------------------------------------------------
# FACTOR-SPECIFIC VISUALIZATION FUNCTION
# ---------------------------------------------------------
def export_factor_specific_details(factors, num_factors=12, threshold=0.075, top_n_cells=10, output_dir='./'):
    """
    Iterate through each factor to generate CCC Networks, Cell-Cell Heatmaps, and detailed LR-Cell Clustermaps.
    """
    print(f"\n[*] Extracting specific interaction networks and heatmaps for {num_factors} factors...")

    for i in range(1, num_factors + 1):
        factor_name = f'Factor {i}'
        network_file = os.path.join(output_dir, f"Network_CCC_{factor_name.replace(' ', '')}.pdf")
        lr_heatmap_file = os.path.join(output_dir, f"Clustermap_Top{top_n_cells}_{factor_name.replace(' ', '')}.pdf")
        cc_heatmap_file = os.path.join(output_dir, f"Clustermap_CC_Pairs_{factor_name.replace(' ', '')}.pdf")

        print(f"  -> Processing {factor_name}...")

        # --- Task A: Plot CCC Network ---
        try:
            c2c.plotting.ccc_networks_plot(
                factors,
                included_factors=[factor_name],
                ccc_threshold=threshold,
                nrows=1,
                panel_size=(10, 10),
                filename=network_file
            )
            plt.close()
        except Exception as e:
            print(f"     [!] Warning (Network): Could not plot {factor_name} (possibly too few interactions).")

        # --- Task B: Plot Top LR-Cell Pairs Clustermap ---
        try:
            lr_cell_product = c2c.analysis.tensor_downstream.get_lr_by_cell_pairs(
                factors,
                lr_label='Ligand-Receptor Pairs',
                sender_label='Sender Cells',
                receiver_label='Receiver Cells',
                order_cells_by='receivers',
                factor=factor_name,
                cci_threshold=None,
                lr_threshold=threshold
            )

            if lr_cell_product is not None and lr_cell_product.size > 0:
                # Keep only the Top N cell pairs
                top_cell_pairs = lr_cell_product.max(axis=0).sort_values(ascending=False).head(top_n_cells).index
                lr_cell_product_filtered = lr_cell_product[top_cell_pairs]
                lr_cell_product_filtered = lr_cell_product_filtered.loc[(lr_cell_product_filtered != 0).any(axis=1)]

                c2c.plotting.loading_clustermap(
                    lr_cell_product_filtered,
                    use_zscore=False,
                    figsize=(14, 6),
                    filename=lr_heatmap_file,
                    cbar_label='Loading Product',
                    yticklabels=True
                )
                plt.close()
        except Exception as e:
            print(f"     [!] Error (LR Heatmap): Failed at {factor_name} - {e}")

        # --- Task C: Plot Cell-Cell Joint Loadings Clustermap (Sender vs Receiver) ---
        try:
            loading_product = c2c.analysis.tensor_downstream.get_joint_loadings(
                factors,
                dim1='Sender Cells',
                dim2='Receiver Cells',
                factor=factor_name
            )

            c2c.plotting.loading_clustermap(
                loading_product.T,
                use_zscore=False,
                figsize=(8, 8),
                filename=cc_heatmap_file,
                cbar_label='Loading Product'
            )
            plt.close()
        except Exception as e:
            print(f"     [!] Error (CC Heatmap): Failed at {factor_name} - {e}")

# =========================================================
# EXECUTION PIPELINE
# =========================================================
print("=== 1. LOAD TENSOR FACTORS ===")
factors = c2c.io.load_tensor_factors(os.path.join(INPUT_DIR, 'Loadings.xlsx'))
context_dict = get_context_dict()

print("\n=== 2. GLOBAL TENSOR VISUALIZATIONS ===")
# Generates Boxplots, Context Clustermap, and LR Clustermap
plot_global_tensor_visualizations(factors, context_dict, OUTPUT_FIGS)

print("\n=== 3. FACTOR-SPECIFIC VISUALIZATIONS ===")
# Generates 12 Networks, 12 LR-Cell Clustermaps, and 12 Sender-Receiver Clustermaps
export_factor_specific_details(factors, num_factors=12, threshold=0.075, top_n_cells=10, output_dir=OUTPUT_FIGS)

print("\n[*] ALL DOWNSTREAM ANALYSES COMPLETED SUCCESSFULLY.")